# n2 · 更多算子 More Ops

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 为 `**`（幂）、`exp`、`log`、`relu`、`tanh`、`/`（除法）推导局部导数（local gradient）。
2. 在 `Value` 类中实现上述算子，并通过有限差分验证正确性。
3. 解释为何 `tanh` 和 `relu` 是神经网络中常用的激活函数（activation function），以及其导数的形式。
4. 理解如何用已有算子组合出新算子（如用 `**` 和 `*` 实现除法）。

> 对应能力：**SH8**


## 概念讲解 Concepts

### 1. 算子局部导数汇总 Local Gradients

对每个新算子，关键步骤是：
1. 写出正向计算公式 $\text{out} = f(\text{self})$。
2. 求 $\frac{d \,\text{out}}{d \,\text{self}}$（局部导数）。
3. 在 `_backward` 中将 `out.grad * 局部导数` 累加到 `self.grad`。

| 算子 | 正向 $\text{out}$ | $\frac{d\,\text{out}}{d\,\text{self}}$ | 说明 |
|------|------|------|------|
| `self ** k` | $x^k$ | $k x^{k-1}$ | $k$ 为常数（int/float） |
| `self.exp()` | $e^x$ | $e^x = \text{out.data}$ | 指数函数导数等于自身 |
| `self.log()` | $\ln x$ | $1/x$ | 要求 $x > 0$ |
| `self.relu()` | $\max(0, x)$ | $1$ if $x>0$ else $0$ | 分段线性，原点不可导 |
| `self.tanh()` | $\tanh(x)$ | $1 - \tanh^2(x)$ | 可用 $\text{out.data}$ 高效计算 |
| `self / other` | $x \cdot y^{-1}$ | 用 `self * other**-1` 复用 | 不需新 `_backward` |

---

### 2. 幂运算 Power `**`

$$\text{out} = x^k \implies \frac{d\,\text{out}}{dx} = k x^{k-1}$$

注意 $k$ 必须是常数（不是 `Value`），否则需要对数微分。

---

### 3. 指数函数 Exponential `exp`

$$\text{out} = e^x \implies \frac{d\,\text{out}}{dx} = e^x = \text{out.data}$$

巧妙之处：$e^x$ 对自身求导还是自身，在 `_backward` 中直接用 `out.data`。

---

### 4. 对数函数 Logarithm `log`

$$\text{out} = \ln x \implies \frac{d\,\text{out}}{dx} = \frac{1}{x} = \frac{1}{\text{self.data}}$$

---

### 5. ReLU 激活函数

$$\text{relu}(x) = \max(0, x) \implies \frac{d}{dx} = \begin{cases} 1 & x > 0 \\ 0 & x \le 0 \end{cases}$$

导数是**分段常数**，在 $x=0$ 处不连续（次梯度 subgradient）。

---

### 6. Tanh 激活函数

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

$$\frac{d}{dx} \tanh(x) = 1 - \tanh^2(x)$$

在 `_backward` 中令 $t = \text{out.data}$，则局部导数为 $1 - t^2$。

---

### 7. 除法作为算子组合

$$a / b = a \cdot b^{-1}$$

因此不需要新的 `_backward`，直接复用 `__mul__` 和 `__pow__`：

```python
def __truediv__(self, other):
    return self * other ** -1
```

这展示了**算子复用**（operator composition）的威力。


## 示例 Worked Example

In [ ]:
# ── 完整 Value 类（含所有算子）───────────────────────────────────────────────
import math


class Value:
    """标量值节点，支持完整算子集合。"""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.6f}, grad={self.grad:.6f})"

    # ── 加法 ─────────────────────────────────────────────────────────────────
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    # ── 乘法 ─────────────────────────────────────────────────────────────────
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    # ── 幂运算  d(x^k)/dx = k*x^(k-1) ───────────────────────────────────────
    def __pow__(self, k):
        assert isinstance(k, (int, float)), "k 必须是常数"
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += (k * self.data ** (k - 1)) * out.grad
        out._backward = _backward
        return out

    # ── 取负 ─────────────────────────────────────────────────────────────────
    def __neg__(self):
        return self * -1

    # ── 减法（= 加 负数）────────────────────────────────────────────────────
    def __sub__(self, other):
        return self + (-other if isinstance(other, Value) else Value(-other))

    # ── 除法（= 乘 倒数，复用已有算子）─────────────────────────────────────
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    # ── 指数  d(e^x)/dx = e^x = out.data ────────────────────────────────────
    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.data * out.grad   # e^x 对自身求导仍是 e^x
        out._backward = _backward
        return out

    # ── 对数  d(ln x)/dx = 1/x ──────────────────────────────────────────────
    def log(self):
        assert self.data > 0, "log 要求输入 > 0"
        out = Value(math.log(self.data), (self,), "log")
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    # ── ReLU  d/dx = 1 if x>0 else 0 ────────────────────────────────────────
    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    # ── Tanh  d/dx = 1 - tanh^2(x) ──────────────────────────────────────────
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    __radd__ = __add__
    __rmul__ = __mul__

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


# ── 演示各算子正向值 ───────────────────────────────────────────────────────
x = Value(0.5)
print("x =", x.data)
print(f"  x**2       = {(Value(0.5)**2).data:.4f}  (应为 0.25)")
print(f"  x.exp()    = {Value(0.5).exp().data:.4f}  (应为 {math.exp(0.5):.4f})")
print(f"  x.log()    = {Value(0.5).log().data:.4f}  (应为 {math.log(0.5):.4f})")
print(f"  x.relu()   = {Value(0.5).relu().data:.4f}  (应为 0.5)")
print(f"  (-x).relu()= {Value(-0.5).relu().data:.4f}  (应为 0.0)")
print(f"  x.tanh()   = {Value(0.5).tanh().data:.4f}  (应为 {math.tanh(0.5):.4f})")
print(f"  1.0/x      = {(Value(1.0)/Value(0.5)).data:.4f}  (应为 2.0)")


In [ ]:
# ── 有限差分梯度验证（所有算子）─────────────────────────────────────────────

def fd(fn, x_val, h=1e-5):
    """中心差分数值梯度."""
    return (fn(x_val + h) - fn(x_val - h)) / (2 * h)


def check(name, fn_val, fn_num, x_val, tol=1e-6):
    """比较 autograd 梯度与数值梯度."""
    xv = Value(x_val)
    y = fn_val(xv)
    y.backward()
    ag = xv.grad
    nd = fd(lambda v: fn_val(Value(v)).data, x_val)
    ok = abs(ag - nd) < tol
    status = "✓" if ok else "✗"
    print(f"  {status} {name:12s} x={x_val:5.2f}  autograd={ag:.6f}  finite-diff={nd:.6f}  err={abs(ag-nd):.2e}")
    assert ok, f"{name} gradient mismatch"


print("梯度验证 Gradient Checks:")
check("x**2",    lambda x: x**2,    None,  2.0)
check("x**3",    lambda x: x**3,    None,  2.0)
check("x**0.5",  lambda x: x**0.5,  None,  4.0)
check("exp(x)",  lambda x: x.exp(), None,  1.0)
check("log(x)",  lambda x: x.log(), None,  2.0)
check("relu(+)", lambda x: x.relu(), None,  1.5)
check("relu(-)", lambda x: x.relu(), None, -0.5)
check("tanh(x)", lambda x: x.tanh(), None, 0.8)
check("1/x",     lambda x: Value(1.0)/x, None, 3.0)

print()
print("所有算子梯度验证通过 All gradient checks passed!")


## 动手 Exercise

In [ ]:
# ── 动手练习：sigmoid 激活函数的导数 ─────────────────────────────────────────
# sigmoid(x) = 1 / (1 + exp(-x))
# sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))
#
# 用已有算子实现 sigmoid，并用有限差分验证梯度。

import math

class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    def __repr__(self):
        return f"Value({self.data:.4f}, grad={self.grad:.4f})"
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def __pow__(self, k):
        out = Value(self.data**k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data**(k-1) * out.grad
        out._backward = _backward
        return out
    def __neg__(self): return self * -1
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1
    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward(): self.grad += out.data * out.grad
        out._backward = _backward
        return out
    __radd__ = __add__
    __rmul__ = __mul__
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()


def sigmoid(x):
    """sigmoid(x) = 1 / (1 + exp(-x))，用已有 Value 算子实现."""
    return Value(1.0) / (Value(1.0) + (-x).exp())


# 测试
for xv in [-2.0, 0.0, 1.0, 2.0]:
    x = Value(xv)
    s = sigmoid(x)
    s.backward()
    expected_val  = 1 / (1 + math.exp(-xv))
    expected_grad = expected_val * (1 - expected_val)
    fd_g = ((1/(1+math.exp(-(xv+1e-5)))) - (1/(1+math.exp(-(xv-1e-5))))) / 2e-5
    ok_val  = abs(s.data - expected_val)  < 1e-9
    ok_grad = abs(x.grad - expected_grad) < 1e-6
    ok_fd   = abs(x.grad - fd_g)         < 1e-6
    status = "✓" if (ok_val and ok_grad and ok_fd) else "✗"
    print(f"{status} x={xv:5.1f}  σ(x)={s.data:.5f}  dσ/dx={x.grad:.5f}  expected={expected_grad:.5f}")

print()
print("sigmoid 梯度验证通过！")
print("公式: σ'(x) = σ(x) · (1 - σ(x))")


## 延伸阅读 Further Reading

1. **Karpathy «The spelled-out intro to neural networks»**：<https://www.youtube.com/watch?v=VMj-3S1tku0> — YouTube 视频，手把手实现 micrograd，强烈推荐。
2. **CS231n Backprop Intuition**：<https://cs231n.github.io/optimization-2/> — 包含各激活函数导数的详细推导。
3. **Goodfellow et al. «Deep Learning»** 第 6 章：前馈网络与激活函数的系统性介绍（可免费在线阅读）。
4. **PyTorch 激活函数文档**：<https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity>


## 关联练习 Related Assignments

```bash
python check.py nanograd-l2
```

> 为 `Value` 类实现 `__pow__`、`exp`、`log`、`relu`、`tanh`、`__truediv__` 等算子，通过梯度验证测试。

> 能力标签：**SH8** · threshold ≥ 0.7
